In [2]:
!pip install python-dotenv

In [ ]:
import os 
from dotenv import load_dotenv
load_dotenv()

In [16]:
from openai import OpenAI
# client = OpenAI(api_key = os.environ['OPENAI_KEY'])
client = OpenAI()
def ask_llm(prompt, model = 'gpt-5-nano', temp=1):
    response = client.chat.completions.create(
        model=model,    
        temperature=temp,
        messages=[{
            'role':'user',
            'content' : prompt
        }] 
    )       
    return response.choices[0].message.content   

In [ ]:
# 1. zero-shot Prompting
# 예시없이 지시사항만 던지는것 - LLM의 기본기능
prompt = "이 문장의 감정을 분류해: '오늘 점심 메뉴가 품절이라 너무 슬퍼.'"
print(ask_llm(prompt,temp=1))

주요 감정: 부정적  
세부 감정: 슬픔(강한 수준), 실망도 포함될 수 있습니다.


In [ ]:
# 2. few-shot Prompting
# 이렇게 하는거야 라고 예시(Shot)를 몇개 보여줘서 성능을 높이는 기술
prompt = """
단어를 이모지로 바꿔줘
사과 -> 🍎
자동차 -> 🚗
고양이 -> 🐱
비행기->
"""
print(ask_llm(prompt,temp=1))

비행기 -> ✈️


In [ ]:
# 3. Chain-of-Thought Prompting  Cot(생각의 사슬)
prompt = """
질문 : 5개의 사과중 2개를 먹고 3개를 더 샀어. 몇개 남았지
사과의 단가는 100원
지급한 금액 : 1000원
총 남은 사과의 개수를 세고 그리고 사과를 구입할때 드는 비용을 
계산해서 거스름돈을 계산해줘
계산은 먹은 사과와 남은 사과를 모두 포함한 금액
마지막 출력은 전체 로직을 점검해서 오류가 있는지 확인하고 결과알려줘
"""
print(ask_llm(prompt,temp=1))

다음과 같이 계산해볼게요. 주어진 값은
- 처음 사과: 5개
- 사과를 먹음: 2개
- 남은 사과(먹고 남긴 후) = 5 - 2 = 3개
- 새로 산 사과: 3개
- 산 사과의 단가: 100원
- 지급한 금액: 1000원

1) 현재 남아 있는 사과의 총 개수
- 먹은 뒤 남은 사과 3개 + 새로 산 사과 3개 = 6개

2) 구입에 드는 비용(요청대로 “먹은 사과와 남은 사과를 모두 포함”한 금액으로 계산)
- 총 포함 사과 수 = 먹은 사과 2개 + 현재 남은 사과 6개 = 8개
- 8개 × 100원 = 800원

3) 거스름돈
- 지급액 1000원 − 총 포함 비용 800원 = 200원

추가로 명확히 해두면 좋은 점
- 일반적으로는 현재 가진 사과(6개) 만 계산해서 거스름돈을 구합니다.
  - 이 경우 총 비용 = 6개 × 100원 = 600원
  - 거스름돈 = 1000원 − 600원 = 400원
- eaten(먹은) 부분은 이미 소비된 것이므로, 이번 거래에서 실제로는 지불 대상이 아닙니다. 따라서 “먹은 사과를 포함한다”는 해석은 재무적으로 혼동의 여지가 있습니다.

결론
- 요청대로 먹은 사과를 포함하면: 남은 사과 6개, 총 포함 비용 800원, 거스름돈 200원.
- 일반적인 해석(현재 갖고 있는 6개를 기준)으로 하면: 거스름돈 400원.

로직 점검 결과 및 오류 여부
- 초기 값과 연산은 일관되나, “먹은 사과를 포함한 비용”은 실제 거래 맥락과 다릅니다. 이 해석은 의도했는지 확인이 필요합니다.
- 남은 사과의 수는 6개로 올바르게 계산됩니다.
- 거스름돈 계산은 포함하는 항목에 따라 200원 또는 400원이 나옵니다. 어떤 해석이 맞는지 의도대로 맞춰 정리하는 것이 좋습니다.

원하시면 특정 해석(예: 현재 가진 6개만 계산)으로 다시 정확한 결과를 정리해 드릴게요.


In [ ]:
# 4. self-Consistency(자기 일관성)
# 한번만 묻지 않고 여러번(예 : 3번) 물어본 뒤 가장 많이 나온 답을 채택함
question = '철수는 학교까지 10분 걸려, 왕복은 몇분 걸릴까?'
answer = []
for _ in range(3):
    answer.append(ask_llm(question))
print(f'수집된 답변들: ', answer)
from collections import Counter
counter = Counter(answer)
counter.most_common(1)

수집된 답변들:  ['20분. 한 방향이 10분이니 왕복은 10분 + 10분 = 20분입니다.', '20분입니다. 한 방향이 10분이므로 왕복은 10분 + 10분 = 20분입니다. (단, 교통 상황에 따라 다를 수 있습니다.)', '20분입니다. 한 방향이 10분이므로 왕복은 10분 + 10분 = 20분이에요.']


Counter({'20분. 한 방향이 10분이니 왕복은 10분 + 10분 = 20분입니다.': 1,
         '20분입니다. 한 방향이 10분이므로 왕복은 10분 + 10분 = 20분입니다. (단, 교통 상황에 따라 다를 수 있습니다.)': 1,
         '20분입니다. 한 방향이 10분이므로 왕복은 10분 + 10분 = 20분이에요.': 1})

In [22]:
# 5. Generate Knowlege Prompting(지식생성)
# 바로 답하지 말고 관련된 지식을 먼저 생성한뒤에 그 지식을 바탕으로 답하게 됨
# 1 지식생성
knowledge = ask_llm('골프라는 스포츠에 대해 사실적인 지식 3가지만 나열해줘')
print(f'[지식] : {knowledge}')
# 2 단계 : 지식을 활용해 답변
prompt = f"""
다음 지식을 참고해서 '골프에서 홀인원이 왜 어려운지' 설명해줘.
모든 답변은 한글로 작성
[지식] : {knowledge}
"""
print(ask_llm(prompt))

[지식] : - 골프 코스는 일반적으로 18홀로 구성되며, 각 홀마다 표준 타수인 파(par)가 정해져 있다. (참고로 9홀 코스도 있어 9홀 또는 두 번 라운드로 18홀을 구성하기도 한다.)

- 점수는 타수로 계산되며, 가장 일반적인 형식은 스트로크 플레이이고, 선수 간 실력 차를 보정하기 위해 핸디캡 제도가 사용된다.

- 프로 골프의 주요 투어로 PGA Tour, DP World Tour, LPGA Tour가 있으며, 메이저 대회로 Masters, US Open, The Open Championship, PGA Championship가 있다.
다음은 주어진 지식을 바탕으로, 왜 골프에서 홀인원이 그렇게 어려운지에 대한 설명입니다.

정의
- 홀인원은 티샷 한 번으로 공이 바로 홀 속으로 들어가는 상황을 말합니다. 보통 파 3홀에서 가장 자주 일어나지만, 파 4나 파 5에서도 가끔 생깁니다.

왜 이렇게 어렵나
- 한 번의 샷에 모든 변수를 맞춰야 함
  - 거리는 물론 방향, 속도, 볼의 스핀까지 완벽하게 맞아야 합니다. 바람, 온도, 습도 같은 기상 조건도 공의 비행 궤도에 큰 영향을 줍니다.
  - 또한 그린의 상태(진동, 속도)와 그린 위의 언덕과 경사도 최종 위치를 크게 바꿉니다. 한 번의 샷으로 컵 입구에 공이 들어가려면 이 모든 요소가 이상적으로 작동해야 합니다.

- 컵의 크기와 목표의 좁음
  - 표준 골프컵의 입구 직경은 약 4.25인치(약 10.8cm)이고, 골프공의 지름은 약 1.68인치(약 4.3cm)입니다. 이 차이는 물리적으로도 “볼이 들어갈 확률이 아주 작은 목표 영역”임을 뜻합니다. 이 좁은 목표를 정확히 맞추는 것이 핵심입니다.

- 코스 구성과 홀의 특성
  - 골프 코스는 18홀로 구성되며, 각 홀마다 파(par)가 정해져 있습니다. 파3 홀은 이론상 한 번의 샷으로 들어갈 가능성이 가장 높아 보이지만, 실제로는 거리와 위험요소(벙커, 워터해저드 등) 때문에 쉽지 않습니다.
  - 파4/파5에서도 홀인원이 나올 수 있지

In [20]:
prompt = '골프에서 홀인원이 왜 어려운지 설명해줘'
print(ask_llm(prompt))

골프에서 홀인원(Hole-in-one)이 어렵다고 느껴지는 주된 이유를 정리해 볼게요.

왜 어려운가 (주요 요인)
- 아주 작은 목표 구멍: 골프 홀의 직경은 약 4.25인치(약 10.8cm)이고, 공의 지름은 약 1.68인치(약 4.3cm)로, 사실상 아주 좁은 목표에 공이 들어가야 해요. 이 작은 차이로도 성공 여부가 갈립니다.
- 거리와 속도의 정밀한 조합 필요: 한 번의 샷에서 정확히 필요한 거리(비거리), 볼의 속도, 스핀을 맞춰야 홀 안으로 들어가고 남은 거리 없이 멈춰야 합니다. 살짝만 빗나가도 홀 안으로 들어가지 않죠.
- 그린의 경사와 속도 변화: 그린은 경사(오르막/내리막), 잔디의 상태, 표면 속도에 따라 볼이 굴러가는 방향과 속도가 크게 달라집니다. 경사 하나만 잘못 읽어도 볼이 그린 밖으로 굴러가거나 홀 밖으로 튀어나갈 수 있습니다.
- 바람과 기상 조건의 영향: 바람은 비거리와 곡선을 크게 바꿉니다. 특히 파3에서 홀인원을 노릴 때 바람의 방향과 세기가 결정에 큰 차이를 만듭니다.
- 샷의 예측 불확실성: 공의 튀김, 핀의 위치, 잔디의 상태(카브, 러프, 러프의 질) 등 예측하지 못한 변수들이 한두 가지 생길 수 있습니다.
- 확률적인 측면: 평균 아마추어 골퍼의 홀인원 확률은 대략 1만 2천 분의 1 정도로 보기도 하고, 프로 선수의 확률은 그보다 더 좋다고 하지만 still 수천 분의 1 수준입니다. 즉, 기술이 뛰어나도 매 샷마다 이 확률에 맞닥뜨리는 셈이고, 많은 라운드에서 한두 번도 나오지 않을 수 있습니다.

간단히 말해
- 홀인원은 샷 하나가 맞물려 들어가야 하는 매우 드문 사건으로, 거리 제어와 방향, 그린 읽기, 바람 등 여러 변수의 완벽한 결합이 필요합니다. 이 조합이 매우 쉽지 않아서 어렵게 느껴집니다.

도움이 될 만한 팁
- 거리 감각과 방향성 훈련: 특정 거리의 샷을 꾸준히 정확히 맞추는 연습을 많이 하세요. 거리별로 로프트를 조정하는 감각을 키우면 도움이 됩니다.
- 그린 읽기 연습: 경사와 속도를 읽는 

In [23]:
# 6. Prompt Chaining(프롬프트 체이닝)
# 복잡한 일을 한번에 시키지 않고 A작업의 결과를 B작업의 입력으로 넘겨주는 파이프라인
# Step 1 : 주체 추출
text = '이메일: 안녕하세요, 이번 주 금요일 회의는 2시로 변경되었습니다.'
topic = ask_llm(f'다음 텍스트에서 핵심 주제만 단어로 뽑아줘 출력은 한글로:{text}')

# step2 : 답장 작성
reply = ask_llm(f"'{topic}'에 대해 '알겠습니다'라는 정중한 답장 메일을 써줘")
print(reply)

다음과 같은 정중한 답장 메일을 사용하시면 좋습니다.

제목: 회의 시간 변경 확인

안녕하세요 [담당자 이름]님,

회의 시간이 금요일 2시로 변경된 것을 확인했습니다. 알겠습니다. 해당 시간에 참석하도록 하겠습니다.

필요하신 자료나 준비사항이 있다면 미리 공유해 주시면 감사하겠습니다. 또한 회의 링크나 장소 등 추가 정보가 있다면 함께 알려주시면 좋겠습니다.

감사합니다.
[이름]
[직함/부서]
[연락처]


In [24]:
# 8 Retrieval Augmented Generation(RAG 검색 증강 생성)
# 이론 : LLM이 모르는 외부 데이터(회사문서등)을 찾아서 (Retrieval)프롬프트에 넣어주고 답하게 함

# 가상의 검색된 문서
retrieved_doc = "문서내용: 우리 회사의 재택근무는 매주 수요일만 가능하다"

prompt = f'''
아래[참조문서]를 기반으로 답변해, 문서에 없으면 모른다고 해.
[참조문서] : {retrieved_doc}
질문 : 재택근무는 언제 할 수 있어?
'''
print(ask_llm(prompt))

재택근무는 매주 수요일에만 가능합니다.


In [25]:
# 9 Automatic Reasoning and Tool-use 자동 추론 및 도구 사용
# LLM이 스스로 계산기나 검색엔진 같은 도구가 필요한지 판단하고 호출형식을 뱉어내는 것
prompt = '''
계산이 필요하면 [CALC: 수식] 이라고 출력해.
질문 : 3452 * 192는 뭐야?
'''
response = ask_llm(prompt)
print(response)


[CALC: 3452 * 192]
662,784


In [ ]:
prompt = """
너는 자동 도구 선택 시스템이야.
다음과 같은 도구를 사용할 수 있어:

1.계산기 -> [CALC : 수식]
2.날씨 조회 ->[WEATHER : 도시명]
3.일반질문 - >직접입력

규칙:
- 계산이 필요하면 [CALC: ...] 출력
- 날씨 정보가 필요하면 [WEATHER: 도시명] 출력
- 그 외 는 일반적인 답변

질문: 3458 * 256의 결과와 
서울의 내일 날씨는 어때?
"""

import re
def process_response(text):
    # 계산기
    calc = re.findall(r'\[CALC :\s*(.*?)\]',text)
    if calc:
        expr = calc[0]
        return eval(expr)
    # 날씨
    weather = re.findall(r'\[WEATHER :\s*(.*?)\]',text)
    if weather:
        city = weather[0]
        return '오늘 늦은 오후까지 곳에 따라 약한 비, 내일 비, 내일 곳에 따라 돌풍.천둥.번개.싸락우박, 내일 강한 바람과 서해상 풍랑 유의'

response = ask_llm(prompt)
print(response)

for res in response.split('\n'):
    process_response(res)

In [ ]:
%pip install requests

In [57]:
import os 
from dotenv import load_dotenv
load_dotenv()
API_KEY = 'fdf46d9b90a6dce13b4e54db8621e743'
lat = 37.25
lon = 126.45
print(API_KEY)
url = 'https://api.openweathermap.org/data/2.5/weather?lat={lat}&lon={lon}&appid={API_KEY}'
import requests
response = requests.get(url)
response

fdf46d9b90a6dce13b4e54db8621e743


<Response [401]>